In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import r2_score, mean_absolute_error

# --- Data and model config (from your selected model) ---
excel_path = Path("../fluoride_modeling_properties_w_experimental_data.xlsx")
sheet_name = "CS2_combined_ddg"
target_col = "ddg_flipped"
set_col    = "set" # keep the original train/val/test split from the spreadsheet
ligand_col = "ligandID"  # for labeling points in plots, if desired

selected_features = [
    "η_max",
    "Sterimol_B1_Ni_N2(Å)_morfeus_low_E",
]

feature_name_map = {
    "η_max": "η (LUMO - HOMO)",
    "Sterimol_B1_Ni_N2(Å)_morfeus_low_E": "Ni−N2 Sterimol B1",
}

if not excel_path.exists():
    raise FileNotFoundError(f"Could not find data file: {excel_path.resolve()}")

df = pd.read_excel(excel_path, sheet_name=sheet_name, header=1).copy()

required_cols = selected_features + [target_col, set_col]
missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise KeyError(f"Missing required columns in {sheet_name}: {missing}")

# Keep only rows with complete data for selected model features/target/set.
work_df = df[required_cols + [ligand_col]].copy()
work_df[target_col] = pd.to_numeric(work_df[target_col], errors="coerce")
for col in selected_features:
    work_df[col] = pd.to_numeric(work_df[col], errors="coerce")
work_df[set_col] = work_df[set_col].astype(str).str.strip().str.lower()
work_df = work_df.dropna(subset=selected_features + [target_col])

X_all = work_df[selected_features].copy()
y_all = work_df[target_col].copy()

train_val_mask = work_df[set_col].isin(["train", "validation"])
test_mask = work_df[set_col].eq("test")

X_train, y_train = X_all.loc[train_val_mask], y_all.loc[train_val_mask]
X_test, y_test = X_all.loc[test_mask], y_all.loc[test_mask]

# Fit your exact model.
model = KernelRidge(alpha=0.1, kernel="rbf")
model.fit(X_train, y_train)

# --- SHAP analysis ---
X_all_unique = X_all.drop_duplicates()

# fit a GAM model to the data
import interpret.glassbox

model_ebm = interpret.glassbox.ExplainableBoostingRegressor(interactions=0)
model_ebm.fit(X_train, y_train)

# explain the GAM model with SHAP
explainer_ebm = shap.Explainer(model_ebm.predict, X_all_unique)
shap_values_ebm = explainer_ebm(X_all_unique)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

# Align ligand_class to the unique rows used for SHAP explanation
ligand_class = df.loc[X_all_unique.index, "ligand_class"].fillna("unknown").astype(str).str.strip().str.lower()

_marker_keys = {"box": "o", "biox": "s", "pyox": "^", "pynx": "^"}

def _get_marker(cls):
    for key, m in _marker_keys.items():
        if key in cls:
            return m
    return "D"

shap_vals = shap_values_ebm.values
feat_vals  = shap_values_ebm.data
n_features = shap_vals.shape[1]
feature_display_names = [feature_name_map.get(f, f) for f in selected_features]

cmap = plt.cm.RdBu_r
fig, ax = plt.subplots(figsize=(7, 2.5))
rng = np.random.default_rng(42)

for feat_idx in range(n_features):
    sv = shap_vals[:, feat_idx]
    fv = feat_vals[:, feat_idx].astype(float)
    fv_min, fv_max = fv.min(), fv.max()
    fv_norm = (fv - fv_min) / (fv_max - fv_min + 1e-12)
    colors = cmap(fv_norm)
    y_jitter = rng.uniform(-0.05, 0.05, size=len(sv))
    y_pos = feat_idx + y_jitter

    for cls in sorted(ligand_class.unique()):
        mask = (ligand_class == cls).values
        if mask.sum() == 0:
            continue
        ax.scatter(
            sv[mask], y_pos[mask],
            c=colors[mask],
            marker=_get_marker(cls),
            s=45, linewidths=0.4, edgecolors="k",
            alpha=0.85,
            label=cls if feat_idx == 0 else "_nolegend_",
            zorder=3,
        )

ax.set_yticks(range(n_features))
ax.set_yticklabels(feature_display_names, fontsize=11)
ax.set_ylim(-0.5, n_features - 0.5)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("SHAP value", fontsize=11)
ax.set_title("SHAP Beeswarm (EBM)", fontsize=12)
ax.spines[["top", "right"]].set_visible(False)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=mcolors.Normalize(vmin=0, vmax=1))
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.02, fraction=0.035)
cbar.set_label("Feature value", fontsize=10)
cbar.set_ticks([0, 1])
cbar.set_ticklabels(["Low", "High"])

legend_handles = [
    plt.scatter([], [], marker=_get_marker(cls), color="grey", s=45,
                edgecolors="k", linewidths=0.4, label=cls)
    for cls in sorted(ligand_class.unique())
]
ax.legend(handles=legend_handles, title="Ligand class",
          loc="lower right", fontsize=9, title_fontsize=9)
plt.tight_layout()
plt.show()